Notebook to generate synthetic goldens which will be (a) reviewed by a human and (b) used as a test set to evaluate the RAG chatbot.  

This notebook requires a Groq API key to generate the goldens.  Note that I used the free model `groq/compund-mini` (which wrapped calls to `llama-3.3-70b-versatile` and `openai/gpt-oss-120b`) to generate these goldens, but this model was deprecated on 21 September 2026.  If you wish to run this code, plesae replace 'groq/compound-mini' with a valid Groq model in cell 4.

Since I used a free model, there are quota-sensitive constraints, such as generating goldens from only a sample of the chunks using `generate_goldens_from_contexts()` rather than using `generate_goldens_from_docs()` and setting limits and time-delays on the number of requests.

Note on the golden dataset: early entries in the goldens CSV were generated during early runs which resampled a small number of source chunks, and these duplicate chunks produced distinct goldens. I retained them because the project's next stage is manual golden review, and these extra goldens remain valid evaluation candidates. This notebook has been corrected to sample chunks correctly before generation, and this version was used for the later and final generation runs.

In [0]:
%pip install -r requirements.txt
dbutils.library.restartPython()

In [0]:
import os
import re
import time
import csv
import json
import hashlib
import random
from typing import List
from pathlib import Path
from collections import defaultdict, Counter
from openai import OpenAI
from deepeval.models import DeepEvalBaseLLM
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from tests.custom_groq_model import CustomGroqModel


In [0]:
DATA_PATH = "fca_cobs_pdfs"
CHUNKS_PATH = "data/chunks.jsonl"
OUTPUT_CSV = "test_sets/synthetic_goldens.csv"
GROQ_MODEL = "groq/compound-mini"

TARGET_PER_CHAPTER = 5
DAILY_QUOTA = 10
PAUSE = 75

In [0]:
# Cleaning function (this is improved over notebook 01, I need to refactor)
SKIP_EXACT = {'R', 'G', 'N', 'E', 'P', 'COBS', 'CHAPTER'}

def is_noise(line: str) -> bool:
    if not line or re.match(r'^[\.\s\-_]+$', line):
        return True
    if line.upper() in SKIP_EXACT:
        return True
    if 'www.handbook.fca.org.uk' in line:
        return True
    if re.search(r'(January|February|March|April|May|June|July|August|'
                 r'September|October|November|December)\s+\d{4}', line):
        return True
    if re.match(r'^(COBS|APP|SUP)\s+\d+[A-Z]?(\.\d+)*$', line, re.I):
        return True
    if len(line) < 60 and re.match(r'^(COBS|APP|SUP)\s+\d+[A-Z]?\s+\w', line, re.I):
        return True
    return False

def clean_fca_text(text: str) -> str:
    blocks = re.split(r'\n\s*\n', text)
    paras = []
    for block in blocks:
        lines = [ln.strip() for ln in block.split('\n')]
        kept = [ln for ln in lines if not is_noise(ln)]
        if kept:
            paras.append(' '.join(kept))
    return '\n\n'.join(paras)

# generate stable chunk id. if text is same, id is same
def generate_chunk_id(doc: Document) -> str:
    text = doc.page_content
    
    # create hash of the text
    hash_obj = hashlib.sha1(text.encode('utf-8'))
    hash_hex = hash_obj.hexdigest()
    
    # get the chapter code (remove spaces for consistency)
    chapter = doc.metadata.get('chapter', 'Unknown').replace(' ', '')
    
    # combine chapter and first 12 chars of hash
    return f"{chapter}-{hash_hex[:12]}"

For generating goldens, I want to include chapter metadata on the chunks (so I can sample of to 5 chunks from each chapter, fewer if the chapter is short), and have longer chunks to provide a reasonable context. 

In [0]:
# check if chunks file exists
if not Path(CHUNKS_PATH).exists():
    print("No frozen chunks found. Running ingestion pipeline")
    
    loader = PyPDFDirectoryLoader(DATA_PATH)
    docs = loader.load()
    print(f"Loaded {len(docs)} pages from PDFs.")

    # add chapter metadata and clean
    docs_with_chapters = []
    for doc in docs:
        source = doc.metadata.get("source", "")
        match = re.search(r'COBS\s+(\d+[A-Z]?)', source)
        
        if match:
            chapter_num = match.group(1)
            chapter_label = f"COBS {chapter_num}"
        else:
            chapter_label = "Unknown"
        
        cleaned_text = clean_fca_text(doc.page_content)
        
        docs_with_chapters.append(Document(
            page_content=cleaned_text,
            metadata={"source": source, "chapter": chapter_label}
        ))

    # merge pages by chapter
    chapter_texts = defaultdict(list)
    for doc in docs_with_chapters:
        chapter_texts[doc.metadata["chapter"]].append(doc.page_content)

    merged_docs = [
        Document(page_content="\n\n".join(texts), metadata={"chapter": chapter})
        for chapter, texts in chapter_texts.items()
    ]
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", ". ", "\n", " ", ""], # prioritise sentence splits
    )
    
    # chunk merged documents
    all_chunks = []
    for doc in merged_docs:
        # split_documents returns a list of Document objects
        doc_chunks = text_splitter.split_documents([doc])
        all_chunks.extend(doc_chunks)

    # filter out short chunks 
    valid_chunks = [c for c in all_chunks if len(c.page_content) >= 800]

    # quick check
    print(f"Total chunks before filtering: {len(all_chunks)}")
    print(f"Total valid chunks (>=800 chars): {len(valid_chunks)}")

    # check distribution by chapter
    chapter_counts = Counter(c.metadata["chapter"] for c in valid_chunks)
    print("\nChunks per chapter:")
    for chapter, count in sorted(chapter_counts.items()):
        print(f"{chapter:10s}: {count} chunks")

    # check shortest valid chunk
    if valid_chunks:
        sorted_by_len = sorted(valid_chunks, key=lambda x: len(x.page_content))
        shortest = sorted_by_len[0]
        print(f"\nShortest valid chunk: {len(shortest.page_content)} chars")
        
    for chunk in valid_chunks:
        chunk.metadata['cid'] = generate_chunk_id(chunk)
    
    # save to JSONL
    with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
        for chunk in valid_chunks:
            line = {"content": chunk.page_content, "metadata": chunk.metadata}
            f.write(json.dumps(line) + "\n")
    print(f"Saved {len(valid_chunks)} chunks to {CHUNKS_PATH}")
else:
    print(f"Loading frozen chunks from {CHUNKS_PATH}")
    # load the frozen file
    valid_chunks = []
    with open(CHUNKS_PATH, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            valid_chunks.append(Document(page_content=d["content"], metadata=d["metadata"]))
    print(f"Loaded {len(valid_chunks)} chunks.")

# check distribution by chapter
chapter_counts = Counter(c.metadata["chapter"] for c in valid_chunks)
print("\nChunks per chapter:")
for chapter, count in sorted(chapter_counts.items()):
    print(f"{chapter:10s}: {count} chunks")

Check the CSV of already-generated goldens against the chunks list.

In [0]:
def analyse_coverage(csv_path="test_sets/synthetic_goldens.csv"):
    """
    Reads the existing CSV and returns:
    1. Counts of goldens per chapter.
    2. A set of 'cid's (chunk IDs) that have already been used.
    """
    counts = Counter()
    used_cids = set()
    
    if not os.path.isfile(csv_path):
        print(f"Note: {csv_path} not found. Starting fresh.")
        return counts, used_cids
    
    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # only count rows that actually have a question
            if row.get("question"):
                counts[row["chapter"]] += 1
                
                # check if this row has a 'cid' column (older rows may not have)
                cid = row.get("cid")
                if cid:
                    used_cids.add(cid)
                    
    return counts, used_cids

current_counts, used_cids = analyse_coverage()

print(f"Current total goldens: {sum(current_counts.values())}")
print(f"Unique CIDs tracked: {len(used_cids)}")

# how many chunks are actually available per chapter?
available = Counter(c.metadata['chapter'] for c in valid_chunks)

print("Coverage vs target:")
for chapter in sorted(available):
    current = current_counts[chapter]
    ceiling = min(TARGET_PER_CHAPTER, available[chapter])   # don't exceed what exists
    need = max(0, ceiling - current)
    status = "DONE" if need == 0 else f"MISSING {need}"
    print(f"{chapter:10s}: {current}/{ceiling} (pool: {available[chapter]}) {status}")

Function to generate which chunks to use to generate goldens for ONE day, maximum of DAILY_QUOTA.

In [0]:
def plan_generation(chunks, counts, used_cids, target=TARGET_PER_CHAPTER, seed=42):
    rng = random.Random(seed)
    
    # group chunks by chapter
    by_chapter = defaultdict(list)
    for c in chunks:
        by_chapter[c.metadata['chapter']].append(c)
    
    # sort chunks by CID for deterministic selection
    for ch in by_chapter:
        by_chapter[ch].sort(key=lambda x: x.metadata['cid'])
    
    todo = []
    remaining_quota = DAILY_QUOTA  
    
    # iterate chapters in sorted order for stability
    for chapter in sorted(by_chapter):
        if remaining_quota <= 0:
            break
            
        current = counts[chapter]
        pool_size = len(by_chapter[chapter])
        ceiling = min(target, pool_size)
        need = max(0, ceiling - current)
        
        if need > 0:
            # filter out chunks whose CID is already used
            available = [c for c in by_chapter[chapter] if c.metadata['cid'] not in used_cids]
            
            # we need 'need' chunks, but we might be low on quota
            # so take min of 'need', 'remaining_quota', and 'available'
            take = min(need, remaining_quota, len(available))
            
            if take > 0:
                selected = rng.sample(available, take)
                todo.extend(selected)
                remaining_quota -= take
            
    return todo

todays_chunks = plan_generation(valid_chunks, current_counts, used_cids)

print(f"Plan: generate {len(todays_chunks)} goldens for today.")
print(f"Remaining quota: {10 - len(todays_chunks)}")
print("\nChapters to cover (with CID filtering):")
chapter_counts = Counter(c.metadata['chapter'] for c in todays_chunks)
if len(chapter_counts) == 0:
    print("    None, all targets met.")
else:
    for ch, count in sorted(chapter_counts.items()):
        print(f"  {ch}: {count} chunk(s)")

Add the Groq API key if necessary.

In [0]:
from getpass import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Initialise the DeepEval Synthesizer.  Note that it uses the CustomGroqModel (see `tests.custom_groq_model.py`), since DeepEval's default is an OpenAI model.  I used the free model `groq/compund-mini` which called `llama-3.3-70b-versatile` and `openai/gpt-oss-120b` but was deprecated on 21 September 2026.

In [0]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ContextConstructionConfig, StylingConfig

synthesizer_llm = CustomGroqModel(
    model=GROQ_MODEL, 
    seconds_delay=60, # slow the rate (free model)
    temperature=0.0,
    max_tokens=1000
)

styling_config = StylingConfig(
    input_format="Questions about the provided context",
    expected_output_format="Direct answer based on the context",
    task="Question Answering",
    scenario="Testing RAG accuracy"
)

synthesizer = Synthesizer(
    model=synthesizer_llm,
    styling_config=styling_config
)

Generate the planned chunks

In [0]:
OUTPUT_CSV = "test_sets/synthetic_goldens.csv"
fieldnames = ["chunk_index", "chapter", "question", "answer", "evolutions",
              "synthetic_input_quality", "context_preview", "context", "cid"]
PAUSE = 75

# pick up chunk_index stating point 
existing_indices = []
if os.path.isfile(OUTPUT_CSV):
    with open(OUTPUT_CSV, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            idx = row.get("chunk_index", "")
            if idx.isdigit():
                existing_indices.append(int(idx))

next_index = max(existing_indices, default=0) + 1
print(f"Starting chunk_index at {next_index}")

if not os.path.isfile(OUTPUT_CSV):
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    new_this_run = 0

    for chunk in todays_chunks:
        cid = chunk.metadata["cid"]
        chapter = chunk.metadata.get("chapter", "Unknown")
        success = False

        for attempt in range(3):
            try:
                goldens = synthesizer.generate_goldens_from_contexts(
                    contexts=[[chunk.page_content]],
                    max_goldens_per_context=1
                )
                if not goldens:
                    print(f"ALERT {cid}: no golden returned")
                    break

                g = goldens[0]
                ctx = " ".join(g.context) if isinstance(g.context, list) else str(g.context)

                writer.writerow({
                    "chunk_index": str(next_index),
                    "chapter": chapter,
                    "question": g.input,
                    "answer": g.expected_output,
                    "evolutions": g.additional_metadata.get("evolutions", ""),
                    "synthetic_input_quality": g.additional_metadata.get("synthetic_input_quality", ""),
                    "context_preview": ctx[:150] + "...",
                    "context": ctx,
                    "cid": cid,
                })
                f.flush()
                os.fsync(f.fileno())
                next_index += 1
                print(f"{new_this_run + 1}/10 {chapter} | {cid}")
                success = True
                new_this_run += 1
                break

            except Exception as e:
                wait = PAUSE * (attempt + 1)
                print(f"ERROR {cid} attempt {attempt + 1}: {e} — waiting {wait}s")
                time.sleep(wait)

        if not success:
            print(f"SKIPPED {cid} after 3 attempts")

        if new_this_run >= 10:
            print("Stopping after 10 — quota safety")
            break

        time.sleep(PAUSE)